# publish_all_helpfiles

This notebook is a Python-native tutorial derived from the MATLAB workflow name, implemented from scratch for `nSTAT-python`.

- Execution group: `full`
- Workflow family: `data`
- Paper DOI: `10.1016/j.jneumeth.2012.08.009`
- PMID: `22981419`
- Help page: `docs/help/examples/publish_all_helpfiles.md`


Notebook source link: [publish_all_helpfiles.ipynb](https://github.com/cajigaslab/nSTAT-python/blob/main/notebooks/publish_all_helpfiles.ipynb)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from nstat.analysis import Analysis
from nstat.cif import CIFModel
from nstat.decoding import DecodingAlgorithms
from nstat.events import Events
from nstat.history import HistoryBasis
from nstat.signal import Covariate
from nstat.spikes import SpikeTrain, SpikeTrainCollection
from nstat.trial import CovariateCollection, Trial, TrialConfig

TOPIC = "publish_all_helpfiles"
FAMILY = "data"
rng = np.random.default_rng(2026)
print(f"Running notebook topic: {TOPIC} (family={FAMILY})")

def validate_numeric_checkpoints(metrics: dict[str, float], limits: dict[str, tuple[float, float]], topic: str) -> None:
    if not metrics:
        raise AssertionError(f"publish_all_helpfiles: CHECKPOINT_METRICS is empty")
    for key, value in metrics.items():
        if not np.isfinite(value):
            raise AssertionError(f"publish_all_helpfiles: metric '{key}' is not finite: {value}")
    for key, (lo, hi) in limits.items():
        if key not in metrics:
            raise AssertionError(f"publish_all_helpfiles: missing checkpoint metric '{key}'")
        value = float(metrics[key])
        if value < float(lo) or value > float(hi):
            raise AssertionError(
                f"publish_all_helpfiles: metric '{key}'={value:.6f} outside [{float(lo):.6f}, {float(hi):.6f}]"
            )
    print(f"Numeric checkpoints for {topic}:", metrics)


In [ ]:
# publish_all_helpfiles: Python-side publish/audit checks for help artifacts.
from pathlib import Path
import yaml

def resolve_repo_root() -> Path:
    candidates = [Path.cwd().resolve()]
    candidates.append(candidates[0].parent)
    candidates.append(candidates[1].parent)
    for root in candidates:
        if (root / "docs" / "help").exists() and (root / "parity").exists():
            return root
    return candidates[0]

repo_root = resolve_repo_root()
help_root = repo_root / "docs" / "help"
example_root = help_root / "examples"

manifest_path = repo_root / "parity" / "example_mapping.yaml"
manifest = yaml.safe_load(manifest_path.read_text(encoding="utf-8"))
topics = [str(row.get("matlab_topic")) for row in manifest.get("examples", []) if row.get("matlab_topic")]

missing_example_pages = []
for topic in topics:
    page = example_root / f"{topic}.md"
    if not page.exists():
        missing_example_pages.append(topic)

help_files = sorted(str(path.relative_to(help_root)) for path in help_root.rglob("*") if path.is_file())
n_md = sum(1 for name in help_files if name.endswith(".md"))
n_html = sum(1 for name in help_files if name.endswith(".html"))

fig, axes = plt.subplots(2, 1, figsize=(9.4, 6.0), sharex=False)
axes[0].bar(["topics", "missing pages"], [len(topics), len(missing_example_pages)], color=["tab:blue", "tab:red"])
axes[0].set_title(f"{TOPIC}: example-page publish audit")
axes[0].set_ylabel("count")

axes[1].bar(["markdown", "html"], [n_md, n_html], color=["tab:green", "tab:orange"])
axes[1].set_title("Help artifact inventory")
axes[1].set_ylabel("count")
plt.tight_layout()
plt.show()

assert len(topics) > 0
assert len(missing_example_pages) == 0

CHECKPOINT_METRICS = {
    "topics_in_manifest": float(len(topics)),
    "missing_example_pages": float(len(missing_example_pages)),
}
CHECKPOINT_LIMITS = {
    "topics_in_manifest": (1.0, 5000.0),
    "missing_example_pages": (0.0, 0.0),
}


In [ ]:
# Execution checkpoints used by CI.
assert TOPIC != "", "Missing topic metadata"
validate_numeric_checkpoints(CHECKPOINT_METRICS, CHECKPOINT_LIMITS, TOPIC)
print("Topic-specific checkpoint for", TOPIC)
print("Notebook checkpoints passed for", TOPIC)


## Next steps

- Compare this notebook with the corresponding MATLAB helpfile output in the validation PDF.
- Use `tools/reports/generate_validation_pdf.py` to run side-by-side parity scoring.
- Refine model assumptions for this specific example until parity thresholds pass.
